# Vérification du téléchargement de l'orthophoto 20cm

Ce notebook vérifie que `download.ortho.download_ortho` interroge correctement le WMS de la Géoplateforme IGN, découpe l'emprise en tuiles si besoin, assemble le résultat en un GeoTIFF, et affiche le résultat sur une carte.

⚠️ Sur l'emprise complète d'une commune, le téléchargement peut représenter plusieurs dizaines/centaines de requêtes WMS et plusieurs centaines de Mo à quelques Go. Par défaut, ce notebook ne télécharge qu'un **échantillon de 200m x 200m** centré sur l'emprise ; le téléchargement complet (section 4) est désactivé par défaut.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities, TARGET_CRS
from download.ortho import download_ortho, _tile_grid, RESOLUTION, MAX_TILE_PIXELS
from download.limites_administratives import fetch_commune_boundary, resolve_bbox

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Grille de tuiles WMS sur l'emprise complète

Chaque tuile correspond à une requête WMS distincte (limite de taille d'image par requête), avant assemblage en mosaïque.

In [ ]:
import geopandas as gpd
from shapely.geometry import box

bbox = resolve_bbox(entity)
sub_bboxes = _tile_grid(bbox, RESOLUTION, MAX_TILE_PIXELS)
print("Nombre de tuiles WMS necessaires :", len(sub_bboxes))

tiles_gdf = gpd.GeoDataFrame(
    {"tile_id": range(len(sub_bboxes))},
    geometry=[box(*b) for b in sub_bboxes],
    crs=TARGET_CRS,
)
entity_gdf = gpd.GeoDataFrame(
    {"nom_entite": [entity.nom]}, geometry=[box(*bbox)], crs=TARGET_CRS
)
boundary = fetch_commune_boundary(entity.code_insee)
tiles_gdf

## 3. Visualisation des emprises

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

tiles_wgs84 = tiles_gdf.to_crs(epsg=4326)
entity_wgs84 = entity_gdf.to_crs(epsg=4326)
boundary_wgs84 = boundary.to_crs(epsg=4326)

m = leafmap.Map()
m.add_gdf(
    tiles_wgs84,
    layer_name="Grille de tuiles WMS",
    style={"color": "purple", "fillOpacity": 0.05},
    info_mode="on_click",
)
m.add_gdf(
    entity_wgs84,
    layer_name="Bbox de telechargement",
    style={"color": "orange", "fillOpacity": 0},
)
m.add_gdf(
    boundary_wgs84,
    layer_name="Limite communale",
    style={"color": "red", "fillOpacity": 0},
)
m.zoom_to_gdf(entity_wgs84)
m

## 4. Téléchargement d'un échantillon (200m x 200m, centré sur l'emprise)

In [ ]:
xmin, ymin, xmax, ymax = bbox
cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
sample_bbox = (cx - 100, cy - 100, cx + 100, cy + 100)

sample_path = download_ortho(sample_bbox, filename="sample.tif")
print("Fichier ecrit :", sample_path)
print("Taille (Ko) :", round(sample_path.stat().st_size / 1024, 1))

## 5. Affichage de l'échantillon téléchargé

Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

m = leafmap.Map()
m.add_raster(str(sample_path), layer_name="Orthophoto (echantillon)")
m

## 6. Téléchargement complet (optionnel)

Passer `DOWNLOAD_FULL_EXTENT = True` pour télécharger l'orthophoto sur toute l'emprise de l'entité (peut représenter de nombreuses requêtes WMS et un temps d'exécution important).

In [ ]:
DOWNLOAD_FULL_EXTENT = False

if DOWNLOAD_FULL_EXTENT:
    full_path = download_ortho(bbox, filename=f"{entity.code_insee}_orthophoto.tif")
    full_path
else:
    print("Telechargement complet desactive (DOWNLOAD_FULL_EXTENT = False).")